# Kinematic Features (Displacement, Velocity, and Acceleration)

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from pose_dynamics.projects.mirror_game.pipeline import *
from pose_dynamics.features.linear_features import (
    extract_kinematics_per_frame,
    summarise_scalar_timeseries,
)

KEYPOINTS_OF_INTEREST = [
    22,          # left ankle
    23,         # right ankle
    16,        # left hand
    17,        # right hand
    5,         # nose
]

N_KEYPOINTS = 38
XYZ_COLS = [f"{axis}{k}" for k in range(N_KEYPOINTS) for axis in ("x", "y", "z")]
FPS = 30.0

PREPROC_PATH = Path(
    "/Users/cartersale/Documents/Pose_Dynamics/projects/mirror_game/data/preprocessed/mirror_game_preprocessed.csv"
)

df_all = pd.read_csv(PREPROC_PATH)
print(df_all.shape)

missing = [c for c in XYZ_COLS if c not in df_all.columns]
if missing:
    raise ValueError(f"Missing pose columns: {missing[:10]}")

def df_to_pose_tensor(df, xyz_cols=XYZ_COLS, n_keypoints=N_KEYPOINTS):
    arr = df[xyz_cols].to_numpy(dtype=float)
    T, F = arr.shape
    assert F == n_keypoints * 3, f"Expected {n_keypoints*3} features, got {F}"
    return arr.reshape(T, n_keypoints, 3)


(455138, 124)


In [8]:
group_cols = ["pair_trial", "party"]
meta_cols = group_cols + ["Pair", "Trial", "Condition", "Leader", "Role"]

features_rows = []

for (pair_trial, party), df_trial in df_all.groupby(group_cols):
    df_trial = df_trial.sort_values("time_s") if "time_s" in df_trial.columns else df_trial.copy()

    coords = df_to_pose_tensor(df_trial)  # (T, 38, 3)

    # subset to keypoints of interest, in fixed order
    kp_indices = KEYPOINTS_OF_INTEREST
    coords_sub = coords[:, kp_indices, :]   # (T, n_sub, 3)

    kin = extract_kinematics_per_frame(coords_sub, fps=FPS)
    speed = kin["speed"]              # (T-1, n_sub)
    accel = kin["acceleration"]       # (T-2, n_sub)

    row = {}

    # meta
    meta = df_trial.iloc[0]
    for col in meta_cols:
        if col in df_trial.columns:
            row[col] = meta[col]

    # per-keypoint stats (collapse over time only, keep keypoints separate)
    for sub_idx, kp_idx in enumerate(kp_indices):
        label = KEYPOINT_MAPPING[kp_idx]  # e.g. "left_wrist"

        speed_stats = summarise_scalar_timeseries(
            speed[:, sub_idx],   # (T-1,)
            axes=(0,),
            prefix=f"{label}_speed",
            stats=("mean", "std", "max"),
        )
        accel_stats = summarise_scalar_timeseries(
            accel[:, sub_idx],   # (T-2,)
            axes=(0,),
            prefix=f"{label}_accel",
            stats=("mean", "std", "max"),
        )

        row.update(speed_stats)
        row.update(accel_stats)

    # optional: global aggregate across all subset keypoints
    row.update(
        summarise_scalar_timeseries(
            speed, axes=(0, 1), prefix="subset_speed", stats=("mean", "std", "max")
        )
    )
    row.update(
        summarise_scalar_timeseries(
            accel, axes=(0, 1), prefix="subset_accel", stats=("mean", "std", "max")
        )
    )

    features_rows.append(row)

df_lin_subset = pd.DataFrame(features_rows)
print(df_lin_subset.shape)
df_lin_subset.head()


(432, 43)


,pair_trial,party,Pair,Trial,Condition,Leader,Role,left_ankle_speed_mean,left_ankle_speed_std,left_ankle_speed_max,...,nose_speed_max,nose_accel_mean,nose_accel_std,nose_accel_max,subset_speed_mean,subset_speed_std,subset_speed_max,subset_accel_mean,subset_accel_std,subset_accel_max
0,P001_T1,P1,1,1,b2b,P1,Leader,0.410793,0.442962,3.489469,...,2.130662,2.046451,3.042832,35.139624,0.335707,0.448957,5.683179,3.278863,5.737186,98.130051
1,P001_T1,P2,1,1,b2b,P1,Follower,0.170406,0.215651,1.232953,...,0.538390,1.040572,0.784249,4.799745,0.264564,0.409370,5.581967,2.404149,4.521600,81.418022
2,P001_T10,P1,1,10,uni,P2,Follower,0.433125,0.441569,2.801322,...,1.003332,1.703677,1.373300,8.893361,0.511422,0.630983,4.937665,4.634589,6.319548,61.682099
3,P001_T10,P2,1,10,uni,P2,Leader,0.419029,0.451187,2.381167,...,1.357544,1.984883,1.745547,14.031736,0.516085,0.669285,6.696017,4.951539,7.166689,72.967835
4,P001_T11,P1,1,11,f2f,P2,Follower,0.409322,0.447004,2.431256,...,1.331100,1.661078,1.550171,9.688385,0.448443,0.565051,5.223746,3.985186,5.733531,62.834636


In [10]:
OUT_PATH = "/Users/cartersale/Documents/Pose_Dynamics/projects/mirror_game/data/features/mirror_game_linear_features_subset.csv"
df_lin_subset.to_csv(OUT_PATH, index=False)
print(f"Saved subset linear features to:\n{OUT_PATH}")

Saved subset linear features to:
/Users/cartersale/Documents/Pose_Dynamics/projects/mirror_game/data/features/mirror_game_linear_features_subset.csv
